# ⚖️ Veredito B1 — o mercado vivo deixa a crônica mais viva?

Notebook de veredito da **Fase 2 do Degrau 1** (B1 Mercado vivo: demanda elástica, pressão de inventário, regimes Markov boom/bust, âncora pública p*). Protocolo **pré-registrado**:

- Corridas **pareadas**: seeds 42, 7, 11, 23 × B1 desligado/ligado, 8 empresas, 120 ticks — só a flag `market_v2` muda.
- Métricas: eventos salientes da crônica por corrida (sem `flows`/`founding`/`end`), viradas de liderança (`rivalry`), amplitude de preços (mediana de max/min por commodity nos frames) e o placar.
- **Regra de aprovação (régua do PRD):** ledger fecha em todas as corridas **E** eventos salientes ON > OFF em ≥ 3 de 4 seeds **E** amplitude mediana de preços ON > OFF **E** ≥ 4 empresas vivas ao fim de cada corrida ON. Fora isso: REPROVADO/AJUSTAR.

Regra do fundador: simulação de verdade roda AQUI, no Kaggle — nunca na VM de desenvolvimento.

In [ ]:
# 1) Fonte do pacote: dataset privado 'pascalcoimbra/mundo-autonomo-src3'
#    (criado a partir do repo com scripts/kaggle_push_kernel.py --bundle-src)
#    O pacote e puro Python: NAO precisa de pip install. Basta src/ no path.
import glob, os, sys, zipfile

cands = glob.glob('/kaggle/input/**/mundo*.zip', recursive=True) + \
        glob.glob('/kaggle/input/*/src/mundo/__init__.py')
if os.path.isdir('/kaggle/input/mundo-autonomo-src3/src'):
    src_root = '/kaggle/input/mundo-autonomo-src3'
elif cands and cands[0].endswith('.zip'):
    zipfile.ZipFile(cands[0]).extractall('/kaggle/working/pkg')
    src_root = '/kaggle/working/pkg'
else:
    raise SystemExit('Fonte do pacote nao encontrada: anexe o dataset '
                     'pascalcoimbra/mundo-autonomo-src3 a este kernel.')
print('pacote em', src_root)
sys.path.insert(0, os.path.join(src_root, 'src'))
import mundo
print('mundo', mundo.__version__, 'importado de', src_root)


## As corridas pareadas
Cada corrida imprime o placar completo no objeto de resultados; a conservação de dinheiro é verificada corrida a corrida a partir dos contadores do motor (`initial_cash + money_in − money_out`).

In [ ]:
# 2) Corridas pareadas: MESMAS seeds, B1 desligado x ligado.
import statistics

from mundo import chain
from mundo.engine import Engine

SEEDS = [42, 7, 11, 23]
TICKS = 120
SALIENT_IGNORED = {"flows", "founding", "end"}


def metrics(eng, res):
    kinds = {}
    for e in res["chronicle"]:
        kinds[e["kind"]] = kinds.get(e["kind"], 0) + 1
    salient = sum(v for k, v in kinds.items() if k not in SALIENT_IGNORED)
    amps = []
    for c in chain.COMMODITIES:
        series = [f["prices"][c] for f in res["frames"]]
        amps.append(max(series) / min(series))
    total = sum(c.cash for c in eng.companies)
    ledger_ok = abs(total - (eng.initial_cash + eng.money_in
                             - eng.money_out)) < 1e-4
    top = res["scoreboard"][0]
    return {
        "salient": salient,
        "rivalry": kinds.get("rivalry", 0),
        "regime_events": kinds.get("market_regime", 0),
        "price_records": kinds.get("price_record", 0),
        "price_amplitude_median": statistics.median(amps),
        "ledger_ok": ledger_ok,
        "top_cash_name": top["name"], "top_cash": top["cash"],
        "max_stage": max(r["stage"] for r in res["scoreboard"]),
        "alive": sum(1 for r in res["scoreboard"] if r["alive"]),
        "supercomputers": int(eng.total_supercomputers),
    }


results = {}
for seed in SEEDS:
    for label, on in (("off", False), ("on", True)):
        eng = Engine(seed=seed, n_companies=8, market_v2=on)
        res = eng.run(TICKS)
        results[(seed, label)] = metrics(eng, res)
        print(f"seed {seed} B1 {label}: pronto")


## O veredito
A tabela compara OFF × ON por seed e a última linha imprime o veredito pela regra pré-registrada. Os eventos `market_regime`/`price_record` só existem com o B1 ligado — eles são parte do ganho de vivacidade, mas **não** contam sozinhos: a régua exige também mais eventos salientes no total e preços com amplitude real.

In [ ]:
# 3) Comparativo e VEREDITO (regra pre-registrada no topo do notebook)
import json

print(f"{'seed':>5} {'metrica':<26} {'B1 off':>10} {'B1 on':>10}")
wins_salient = 0
amps_off, amps_on = [], []
ledger_all = True
alive_ok = True
for seed in SEEDS:
    off, on = results[(seed, "off")], results[(seed, "on")]
    ledger_all = ledger_all and off["ledger_ok"] and on["ledger_ok"]
    alive_ok = alive_ok and on["alive"] >= 4
    if on["salient"] > off["salient"]:
        wins_salient += 1
    amps_off.append(off["price_amplitude_median"])
    amps_on.append(on["price_amplitude_median"])
    for key in ("salient", "rivalry", "price_amplitude_median"):
        print(f"{seed:>5} {key:<26} {off[key]:>10.2f} {on[key]:>10.2f}")
    print(f"{seed:>5} {'regime+record (so on)':<26} {'-':>10} "
          f"{on['regime_events'] + on['price_records']:>10.0f}")
    print(f"{seed:>5} {'placar: lider de caixa':<26} "
          f"{off['top_cash_name'][:10]:>10} {on['top_cash_name'][:10]:>10}")
print()
print(f"amplitude mediana de precos: off {statistics.median(amps_off):.2f}x "
      f"| on {statistics.median(amps_on):.2f}x")
print(f"eventos salientes: B1 on vence em {wins_salient}/{len(SEEDS)} seeds")
print(f"ledger fecha em todas as corridas: {ledger_all}")
print(f"corridas B1 on com >= 4 empresas vivas: {alive_ok}")

approved = (ledger_all and alive_ok and wins_salient >= 3
            and statistics.median(amps_on) > statistics.median(amps_off))
print()
print("VEREDITO B1:", "APROVADO — a cronica fica visivelmente mais viva "
      "com o mercado vivo e o ledger fecha." if approved else
      "REPROVADO/AJUSTAR — ver metricas acima: a regua do PRD nao foi "
      "atingida; calibrar parametros antes de qualquer nova tentativa.")
with open("/kaggle/working/veredito-b1.json", "w", encoding="utf-8") as f:
    json.dump({f"{s}-{l}": m for (s, l), m in results.items()}, f,
              ensure_ascii=False, indent=1)
print("detalhes em /kaggle/working/veredito-b1.json")
